# Rhythm-level model: NSR / AFib / AFlutter

The second model in this project. `ecg_model.pkl` (trained in `main.ipynb`)
classifies **one beat** as normal or ectopic. This one classifies **one minute
of beats** as sinus rhythm, atrial fibrillation or atrial flutter, which is a
question no single beat can answer. The two run side by side.

Input is `windows_60s.parquet`, built by `icentia_windows.py` from the
Icentia11k subset: one row per 1-minute window, with RR-derived features and a
rhythm label taken by majority overlap with the annotated rhythm spans.

Structure follows `main.ipynb` -- same six classifier families, same
comparison table, best by Macro F1 gets saved. Two things differ:

* **The split is grouped by patient.** Windows from one person share a heart
  and an electrode, so they are not independent observations. A plain
  `train_test_split` would put the same patient on both sides and report a
  score that will not hold on a new person.
* **A decision threshold is chosen** rather than trusting `argmax`, because
  this training set is 54% AFib by construction and a real wearer is not.

In [ ]:
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score)

In [ ]:
windows = pd.read_parquet('windows_60s.parquet')
print(f"Loaded {windows.shape[0]:,} windows from {windows['patient_id'].nunique():,} patients")

# The window table ships unfiltered so thresholds stay tunable here rather than
# in Colab. These are the defaults documented in icentia_windows.py:
#   n_beats     -- at least 30 beats in the minute (60s at HR 30)
#   frac_covered-- at least half the minute carries a rhythm annotation
#   purity      -- at least 70% of the annotated time is the winning rhythm
#   n_drr_used  -- at least one adjacent accepted RR pair, so rmssd/pnn50 exist
dataset = windows.query(
    "n_beats >= 30 and frac_covered >= 0.5 and purity >= 0.7 and n_drr_used >= 1"
).reset_index(drop=True)
dataset['label'] = dataset['label'].astype(str)

print(f"After quality filter: {dataset.shape}")
print(dataset['label'].value_counts())
print("\nNote: this class mix is an artifact of how the subset was selected "
      "(AFib-burden first). It is not population prevalence.")

In [ ]:
FEATURES = ['mean_rr', 'hr', 'hrv_sdnn', 'rmssd', 'pnn50', 'cv_rr', 'rr_entropy']
CLASSES = ['NSR', 'AFib', 'AFlutter']
AF = ['AFib', 'AFlutter']

# mean_rr / hr / hrv_sdnn are this project's rr_interval / heart_rate / hrv,
# measured over a minute instead of over one beat. The other four are the
# standard AFib irregularity measures; amplitude and qrs_width are not here
# because they need the raw waveform and describe beat shape, not rhythm.
X = dataset[FEATURES]
y = dataset['label']
groups = dataset['patient_id'].astype(str)

# Grouped split: no patient may appear in both train and test.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))

X_train_full, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train_full, y_test = y.iloc[train_idx], y.iloc[test_idx]
g_train = groups.iloc[train_idx]

print(f"Train: {len(X_train_full):,} windows / {g_train.nunique():,} patients")
print(f"Test:  {len(X_test):,} windows / {groups.iloc[test_idx].nunique():,} patients")
print(f"Patients in both: {len(set(g_train) & set(groups.iloc[test_idx]))}")

# An RBF SVM is O(n^2) in samples, so it cannot be fitted on 573k rows -- it
# would take hours. Instead of giving the SVM an easier task than the others,
# all six are compared on the same capped sample. Capping PER PATIENT rather
# than taking whole patients keeps all 1,582 training patients represented,
# which matters more for generalising than raw row count.
WINDOWS_PER_PATIENT = 25
rng = np.random.default_rng(42)
positions = pd.Series(np.arange(len(y_train_full)))
sample_idx = np.sort(np.concatenate([
    rng.choice(p.to_numpy(), min(len(p), WINDOWS_PER_PATIENT), replace=False)
    for _, p in positions.groupby(g_train.to_numpy())
]))
X_train, y_train = X_train_full.iloc[sample_idx], y_train_full.iloc[sample_idx]
print(f"\nComparison set: {len(X_train):,} windows from "
      f"{g_train.iloc[sample_idx].nunique():,} patients")
print(y_train.value_counts())

In [ ]:
# Scale-sensitive models (SVM, KNN, Logistic Regression) get a StandardScaler.
# Trees and Naive Bayes don't need it, but a pipeline keeps everything uniform.
# class_weight='balanced' wherever supported, since the class mix is manufactured.
models = {
    'Random Forest': Pipeline([
        ('clf', RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                       n_jobs=-1, random_state=42))
    ]),
    'Decision Tree': Pipeline([
        ('clf', DecisionTreeClassifier(class_weight='balanced', random_state=42))
    ]),
    'Naive Bayes': Pipeline([
        ('clf', GaussianNB())
    ]),
    'SVM (RBF kernel)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', class_weight='balanced', random_state=42))
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=5, n_jobs=-1))
    ]),
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000,
                                   random_state=42))
    ]),
}

# AF-vs-NSR is the question the device actually asks, so it is tracked alongside
# the 3-class metrics: mixing up AFib and AFlutter matters far less than missing
# an abnormal rhythm altogether.
af_true = y_test.isin(AF)
results = []
trained_models = {}

for name, pipe in models.items():
    t0 = time.time()
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    per_class = f1_score(y_test, y_pred, average=None, labels=CLASSES, zero_division=0)
    af_pred = pd.Series(y_pred, index=y_test.index).isin(AF)

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Macro F1': f1_score(y_test, y_pred, average='macro', labels=CLASSES,
                             zero_division=0),
        'NSR F1': per_class[0],
        'AFib F1': per_class[1],
        'AFlutter F1': per_class[2],
        'AF Recall': recall_score(af_true, af_pred, zero_division=0),
        'AF Precision': precision_score(af_true, af_pred, zero_division=0),
        'Seconds': time.time() - t0,
    })
    trained_models[name] = pipe

    print(f"\n=== {name} ===  ({results[-1]['Seconds']:.1f}s)")
    print(confusion_matrix(y_test, y_pred, labels=CLASSES))

results_df = pd.DataFrame(results).sort_values('Macro F1', ascending=False).reset_index(drop=True)
print("\n\n=== Model Comparison (sorted by Macro F1) ===")
print(results_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

In [ ]:
# Macro F1 weights all three classes equally, which is the fair choice when the
# class mix is an artifact of selection. Cross-check AF Recall before accepting
# the winner -- missing an abnormal rhythm is the costly error for a screener.
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]
print(f"Best on the comparison set: {best_model_name}")

# The winner gets refitted on the full training set if it can scale, so the
# shipped model is not limited by the SVM's cost.
SCALES_TO_FULL = {'Random Forest', 'Decision Tree', 'Naive Bayes', 'Logistic Regression'}
if best_model_name in SCALES_TO_FULL:
    t0 = time.time()
    best_model = models[best_model_name]
    best_model.fit(X_train_full, y_train_full)
    print(f"Refitted on all {len(X_train_full):,} training windows ({time.time()-t0:.1f}s)")
else:
    print(f"{best_model_name} does not scale to the full training set; "
          f"keeping the comparison-set fit.")

y_pred = best_model.predict(X_test)
print(f"\n=== {best_model_name} on held-out patients ===")
print(classification_report(y_test, y_pred, labels=CLASSES, digits=4, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=CLASSES)
print("Confusion matrix (rows = true, cols = predicted)")
print(pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_string())

# AFlutter is organised atrial activity, so it sits between NSR and AFib on
# every RR-variability measure. Confusion with AFib is expected and is reported
# separately from the error that actually matters.
n_af = cm[1].sum() + cm[2].sum()
print(f"\nAFib <-> AFlutter mix-ups: {cm[1,2]+cm[2,1]:,} of {n_af:,} AF windows "
      f"({100*(cm[1,2]+cm[2,1])/n_af:.1f}%)")
print(f"AF missed as NSR (the costly error): {cm[1,0]+cm[2,0]:,} of {n_af:,} "
      f"({100*(cm[1,0]+cm[2,0])/n_af:.1f}%)")

if hasattr(best_model.named_steps['clf'], 'feature_importances_'):
    print("\nFeature importance:")
    print(pd.Series(best_model.named_steps['clf'].feature_importances_,
                    index=FEATURES).sort_values(ascending=False).to_string())

## Choosing the alarm threshold

`argmax` implicitly assumes the deployment data looks like the training data.
It does not: this set is 54% AFib because it was selected that way, while a
person wearing the sensor is almost always in sinus rhythm.

Test-set precision is therefore optimistic. The last column below rescales it
to a 2% AF prevalence using the likelihood ratio,
`precision = p*TPR / (p*TPR + (1-p)*FPR)`, which is what a wearer would
actually experience. Pick the serving threshold from that column, not from the
default 0.5.

In [ ]:
if hasattr(best_model, 'predict_proba'):
    proba = best_model.predict_proba(X_test)
    cls = list(best_model.classes_)
    af_score = sum(proba[:, cls.index(c)] for c in AF)

    af_auc = float(roc_auc_score(af_true, af_score))
    print(f"AF-vs-NSR ROC AUC: {af_auc:.4f}  "
          f"(ranking quality, independent of prevalence)")

    DEPLOY_AF_PREVALENCE = 0.02
    print(f"\n{'Threshold':>10s} {'AF Recall':>10s} {'Precision':>10s} {'Prec @2% AF':>12s}")
    for t in (0.5, 0.7, 0.8, 0.9, 0.95):
        pr = af_score >= t
        tpr = (pr & af_true).sum() / af_true.sum()
        fpr = (pr & ~af_true).sum() / (~af_true).sum()
        p = DEPLOY_AF_PREVALENCE
        den = p * tpr + (1 - p) * fpr
        print(f"{t:>10.2f} {tpr:>10.4f} "
              f"{precision_score(af_true, pr, zero_division=0):>10.4f} "
              f"{(p*tpr/den if den else 0):>12.4f}")
    print(f"\nTest-set AF prevalence: {af_true.mean():.1%}")
else:
    af_auc = float('nan')
    print(f"{best_model_name} has no predict_proba; threshold tuning unavailable")

In [ ]:
# Saved as a dict, not a bare pipeline, because the feature ORDER and the WINDOW
# LENGTH are part of the contract. hrv_sdnn here is the SD of ~95 RR intervals
# over 60 seconds, whereas live_predict.py currently computes the SD of the last
# 5 intervals over a 5-second buffer -- feeding that to this model would be
# silently wrong. The expectations travel with the file.
best_model.fit(X, y)   # final fit on every window; evaluation is already done

bundle = {
    'model': best_model,
    'model_name': best_model_name,
    'features': FEATURES,
    'classes': CLASSES,
    'window_sec': 60,
    'fs_trained_on': 250,
    'rr_range_s': [0.24, 2.0],
    'af_auc': af_auc,
    'training_prevalence': {c: float((y == c).mean()) for c in CLASSES},
    'note': ('Serve with a 60s beat-timestamp history. RR is in seconds, so the '
             'sensor sampling rate need not be 250 Hz, but the window length '
             'must match. Prevalence is selection-biased: apply a threshold '
             'rather than argmax.'),
}

joblib.dump(bundle, 'rhythm_model.pkl')
results_df.to_csv('rhythm_model_comparison.csv', index=False)
print(f"Best model: {best_model_name} "
      f"(Macro F1 = {results_df.iloc[0]['Macro F1']:.4f}, AF AUC = {af_auc:.4f})")
print("Saved as rhythm_model.pkl (+ rhythm_model_comparison.csv)")